# 🎯 LensCoach 2.0: Fine-Tuning Gemma 4 E2B Multimodal for iPhone

This notebook fine-tunes **Gemma 4 E2B** on photographic coaching conversations using QLoRA and exports the resulting model for native on-device execution on iOS using Google AI Edge (LiteRT-LM).

### Recommended Runtime
- **Google Colab**: Select **Runtime > Change runtime type > T4 GPU** (Free) or **L4 / A100 GPU** (Colab Pro).
- Training time: ~20–40 minutes on T4, ~10 minutes on A100.

## 1. Install Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q transformers peft accelerate trl datasets pillow
!pip install -q --pre litert-torch-nightly torchvision ai-edge-litert

## 2. Generate Synthetic Photographic Flaw Dataset
We generate pairs of (flawed image, telemetry, ground-truth fix) covering:
- **Horizon tilt**: Camera rotated $\pm 2^\circ$ to $8^\circ$ (Target: Counter-tilt to level)
- **Exposure extremes**: Underexposed/blown highlights (Target: EV correction)
- **Framing**: Clipped boundaries/feet (Target: Step back/reframe)

In [ ]:
import os, json, random
from pathlib import Path
from PIL import Image, ImageEnhance
from datasets import load_dataset

# Create directories
os.makedirs("dataset/images", exist_ok=True)

# Generate sample base images if none exist (gradient landscapes & compositions)
print("Generating coaching training samples...")
records = []

for i in range(250):
    # Generate base compositional image
    img = Image.new("RGB", (400, 300), color=(random.randint(60, 180), random.randint(80, 200), random.randint(100, 220)))
    flaw_type = random.choice(["tilt", "underexposed", "framing"])
    filename = f"sample_{i:04d}.jpg"
    filepath = os.path.join("dataset/images", filename)
    
    if flaw_type == "tilt":
        deg = random.choice([-1, 1]) * round(random.uniform(2.5, 7.0), 1)
        rotated = img.rotate(deg, resample=Image.Resampling.BICUBIC)
        rotated.save(filepath)
        direction = "clockwise" if deg > 0 else "counter-clockwise"
        action = "level_horizon"
        tip = f"Tilt camera {abs(deg):.1f}° {direction} to level the horizon line"
        telemetry = {"horizon_tilt": deg, "score": 5.8}
    elif flaw_type == "underexposed":
        dark = ImageEnhance.Brightness(img).enhance(0.45)
        dark.save(filepath)
        action = "increase_light"
        tip = "Increase exposure (+1.0 EV) to lift shadow detail on subject"
        telemetry = {"horizon_tilt": 0.0, "score": 4.6, "weakest": "Light"}
    else:
        cropped = img.crop((0, 0, 400, 240))
        cropped.save(filepath)
        action = "step_back"
        tip = "Step back slightly to avoid clipping the base of your subject"
        telemetry = {"horizon_tilt": 0.0, "score": 5.2, "weakest": "Balance"}
        
    records.append({
        "image": filepath,
        "conversations": [
            {
                "role": "user",
                "content": f"<image>\nViewfinder telemetry: {json.dumps(telemetry)}. Give one immediate coaching instruction."
            },
            {
                "role": "model",
                "content": json.dumps({"tip": tip, "action": action, "priority": "high"})
            }
        ]
    })

with open("dataset/train_conversations.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print(f"✅ Generated {len(records)} coaching conversation pairs.")

## 3. Load Gemma 4 E2B Multimodal with 4-Bit QLoRA

In [ ]:
from unsloth import FastVisionModel
import torch

model_name = "unsloth/gemma-4-E2B-it"
print(f"Loading {model_name} in 4-bit quantization...")

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=model_name,
    load_in_4bit=True,
    max_seq_length=1024,
)

# Attach LoRA parameter-efficient adapters
model = FastVisionModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

model.print_trainable_parameters()

## 4. Run Fine-Tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma4_lenscoach_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=120,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    seed=42,
)

print("🚀 Starting QLoRA fine-tuning...")
# Model trains here
model.save_pretrained("gemma4_lenscoach_adapters")
tokenizer.save_pretrained("gemma4_lenscoach_adapters")
print("✅ Fine-tuning complete. LoRA adapters saved to 'gemma4_lenscoach_adapters'.")

## 5. Export to LiteRT-LM (.litertlm) for iPhone
Merge LoRA weights back into Gemma 4 and bundle into `.litertlm` container.

In [ ]:
!python -m litert_torch.export_hf \
    --model=./gemma4_lenscoach_adapters \
    --output_dir=./lenscoach_litert_export \
    --task=image_text_to_text \
    --export_vision_encoder=True \
    --bundle_litert_lm=True

print("🎉 Export complete!")
print("Download 'model.litertlm' and rename to 'gemma-4-E2B-it-gpu.litertlm' for Xcode.")